In [4]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(url)

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [5]:
df.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='str')

In [6]:
df = df[
    [
        "engine_displacement",
        "horsepower",
        "vehicle_weight",
        "model_year",
        "fuel_efficiency_mpg",
    ]
]

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   engine_displacement  10000 non-null  int64  
 1   horsepower           9123 non-null   float64
 2   vehicle_weight       10000 non-null  int64  
 3   model_year           10000 non-null  int64  
 4   fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 390.8 KB


In [8]:
# Q1: horsepower

In [9]:
# Q2: 254

In [10]:
df["horsepower"].median()

np.float64(254.0)

In [11]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [12]:
len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [13]:
def train_linear_regression(X, y):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv @ X.T @ y
    return w

In [14]:
def rmse(y, y_pred):
    error = y - y_pred
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [15]:
features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year"
]

target = "fuel_efficiency_mpg"

X_train = df_train[features].copy()
y_train = df_train[target].values

X_val = df_val[features].copy()
y_val = df_val[target].values

In [16]:
X_train_zero = X_train.fillna(0)
X_val_zero = X_val.fillna(0)

w = train_linear_regression(
    X_train_zero.values,
    y_train
)

X_val_zero_matrix = np.column_stack([
    np.ones(X_val_zero.shape[0]),
    X_val_zero.values
])

y_pred = X_val_zero_matrix @ w

rmse_zero = rmse(y_val, y_pred)

round(rmse_zero, 3)

np.float64(2.205)

In [17]:
horsepower_mean = X_train["horsepower"].mean()

X_train_mean = X_train.fillna(horsepower_mean)
X_val_mean = X_val.fillna(horsepower_mean)

w = train_linear_regression(
    X_train_mean.values,
    y_train
)

X_val_mean_matrix = np.column_stack([
    np.ones(X_val_mean.shape[0]),
    X_val_mean.values
])

y_pred = X_val_mean_matrix @ w

rmse_mean = rmse(y_val, y_pred)

round(rmse_mean, 3)

np.float64(2.202)

In [18]:
# Q3: With mean

In [19]:
r = [0, 0.01, 0.1, 1, 5, 10, 100]

In [20]:
X_train = df_train[features].fillna(0)
X_val = df_val[features].fillna(0)

y_train = df_train[target].values
y_val = df_val[target].values

In [21]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [22]:
X_train = df_train[features].fillna(0)
X_val = df_val[features].fillna(0)

y_train = df_train[target].values
y_val = df_val[target].values

In [23]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(
        X_train.values,
        y_train,
        r
    )

    y_pred = w0 + X_val.values.dot(w)

    score = rmse(y_val, y_pred)

    print(r, round(score, 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [25]:
scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]

    X_train = df_train[features].fillna(0)
    X_val = df_val[features].fillna(0)

    y_train = df_train[target].values
    y_val = df_val[target].values

    w0, w = train_linear_regression_reg(
        X_train.values,
        y_train,
        r=0
    )

    y_pred = w0 + X_val.values.dot(w)

    score = rmse(y_val, y_pred)
    scores.append(score)

std = np.std(scores)
print(std)

0.028781274078187116


In [ ]:
# Q5 0.029